[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch11-hw-acceleration/04-tu-kernel-toi-ca-he-thong.ipynb)

# Từ một kernel nhanh tới cả hệ thống: Amdahl, năng lượng, nhiều chip và cacbon

Ba sổ tay trước nhìn vào bên trong một bộ tăng tốc (accelerator). Sổ tay này lùi ra xa. Một kernel nhanh hơn 100
lần chưa chắc làm cả ứng dụng nhanh hơn bao nhiêu, một phép tính rẻ chưa chắc là một kết quả rẻ,
và tám GPU chưa chắc nhanh gấp tám một GPU. Mỗi câu ấy có một phép tính nhỏ đứng sau, và chương
đưa đủ số để làm từng phép.

**Bạn sẽ làm:**
1. tính khoảng cách mà định nghĩa 1.1 nêu, và kịch bản khoảng cách hệ thống của hình 3;
2. dùng định luật Amdahl để xem vì sao 247 lần trên đường nhân ma trận chỉ còn 18.6 lần cho cả ứng
   dụng;
3. tính năng lượng của việc di chuyển dữ liệu so với năng lượng của phép tính;
4. lập lịch đệm kép như một bài toán hai tài nguyên;
5. chạy thật một AllReduce vòng, đếm byte, rồi tính kịch bản tám GPU của sách;
6. tính lại napkin math 1.10 về năng lượng và cacbon, rồi thử độ nhạy của nó.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Khoảng cách mà chương bắt đầu từ đó

Định nghĩa 1.1 đặt A100 cạnh một CPU tham chiếu. Mốc CPU 1 tới 2 TFLOP/s là mốc minh hoạ của chương,
không phải số đo của một con chip cụ thể.

In [ ]:
R_A100 = sach(312, nguon="312 TFLOP/s · 1 tới 2 TFLOP/s · 156 tới 312 lần · 54.2 tỷ")
cpu_thap = sach(1, nguon="312 TFLOP/s · 1 tới 2 TFLOP/s · 156 tới 312 lần · 54.2 tỷ")
cpu_cao = sach(2, nguon="312 TFLOP/s · 1 tới 2 TFLOP/s · 156 tới 312 lần · 54.2 tỷ")
theo_sach("khoảng cách với CPU 2 TFLOP/s", R_A100 / cpu_cao, sach=156,
          nguon="312 TFLOP/s · 1 tới 2 TFLOP/s · 156 tới 312 lần · 54.2 tỷ")
theo_sach("khoảng cách với CPU 1 TFLOP/s", R_A100 / cpu_thap, sach=312,
          nguon="312 TFLOP/s · 1 tới 2 TFLOP/s · 156 tới 312 lần · 54.2 tỷ")

Hình 3 vẽ một kịch bản, sách nói rõ là giả định chứ không phải dự báo: sức tính của một GPU tăng
khoảng 1.7 lần mỗi năm, nhu cầu của mô hình tăng khoảng 6 lần mỗi năm, cả hai chuẩn hoá về năm
2012. Khoảng cách giữa hai đường nới ra theo tỉ số của hai tốc độ.

In [ ]:
cung = sach(1.7, nguon="2017 · 1.7 · 6 · 3 tới 4 · 3–4 · 2012")
cau = sach(6, nguon="2017 · 1.7 · 6 · 3 tới 4 · 3–4 · 2012")
nam_goc = sach(2012, nguon="2017 · 1.7 · 6 · 3 tới 4 · 3–4 · 2012")
gap_moi_nam = cau / cung
gap_thap = sach(3, nguon="2017 · 1.7 · 6 · 3 tới 4 · 3–4 · 2012")
gap_cao = sach(4, nguon="2017 · 1.7 · 6 · 3 tới 4 · 3–4 · 2012")
assert gap_thap <= gap_moi_nam <= gap_cao          # "3 tới 4 lần mỗi năm"
print(f"khoảng cách nới {gap_moi_nam:.2f} lần mỗi năm, nằm trong khoảng 3 tới 4 của sách")
for nam in (1, 3, 5):
    print(f"sau {nam} năm kể từ {nam_goc}: nhu cầu × {cau ** nam:,.0f}, cung × {cung ** nam:.1f},"
          f" khoảng cách × {gap_moi_nam ** nam:,.0f}")
assert gap_moi_nam ** 5 > 500

Sau năm năm của kịch bản, khoảng cách đã hơn 500 lần. Chờ chip nhanh hơn không lấp được nó, nên
phần còn lại của chương là về những cách khác: chuyên biệt hoá, dùng lại dữ liệu, và nhiều chip.
Nhưng chuyên biệt hoá chỉ tăng tốc **một phần** của công việc, và đó là chỗ định luật Amdahl vào.

## 2. Định luật Amdahl: phần không được tăng tốc đặt trần

Nếu một phần $p$ của thời gian ban đầu được tăng tốc $G$ lần, còn phần $1 - p$ giữ nguyên, thì cả
ứng dụng nhanh lên

$$S(p, G) = \frac{1}{(1 - p) + p / G}.$$

Khi $G$ lớn vô hạn, $S$ tiến tới $1/(1 - p)$. Định lý 1.1 của sách lấy ví dụ $p = 0.9$.

**Dự đoán:** với $p = 0.95$ và $G = 247$, cả ứng dụng nhanh hơn bao nhiêu lần? Gần 247, gần 100,
hay gần 20?

In [ ]:
def amdahl(p, G):
    return 1 / ((1 - p) + p / G)


p_dl = sach(0.9, nguon="10 phần trăm · 0.9 · 10 lần")
theo_sach("trần khi p = 0.9", 1 / (1 - p_dl), sach=10, nguon="10 phần trăm · 0.9 · 10 lần")

TOPS_H100 = sach(1979, nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
TOPS_CPU = sach(8, nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
theo_sach("H100 so với CPU trên đường nhân ma trận", TOPS_H100 / TOPS_CPU, sach=247,
          nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
G = sach(247, nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
p_resnet = sach(0.95, nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
theo_sach("ResNet-50: phần không tăng tốc", 1 - p_resnet, sach=0.05,
          nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
theo_sach("ResNet-50: p / G", p_resnet / G, sach=0.0038,
          nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")
theo_sach("ResNet-50: tăng tốc cả ứng dụng", amdahl(p_resnet, G), sach=18.6,
          nguon="247 · 1,979 TOPS · 8 TOPS · 0.95 · 95 · 5 · 0.05 · 0.0038 · 18.6")

p_gpt2 = sach(0.80, nguon="0.80 · 20 · 0.0032 · 4.9 · 5 lần")
theo_sach("GPT-2: phần không tăng tốc (%)", 100 * (1 - p_gpt2), sach=20, nguon="0.80 · 20 · 0.0032 · 4.9 · 5 lần")
theo_sach("GPT-2: p / G", p_gpt2 / G, sach=0.0032, nguon="0.80 · 20 · 0.0032 · 4.9 · 5 lần")
theo_sach("GPT-2: tăng tốc cả ứng dụng", amdahl(p_gpt2, G), sach=4.9, nguon="0.80 · 20 · 0.0032 · 4.9 · 5 lần")
theo_sach("GPT-2: trần khi G vô hạn", 1 / (1 - p_gpt2), sach=5, nguon="0.80 · 20 · 0.0032 · 4.9 · 5 lần")

Đây là kịch bản minh hoạ của sách ("Assume"), không phải số đo. Cùng một con chip, ResNet-50 được
18.6 lần còn GPT-2 sinh token chỉ được 4.9 lần, vì phần việc ngoài đường nhân ma trận của GPT-2
lớn gấp bốn. Hình dưới vẽ $S$ theo $G$ cho ba giá trị $p$.

In [ ]:
cac_G = np.logspace(0, 4, 300)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for p, mau, nhan in ((p_resnet, "#2a78d6", "p = 0.95 (ResNet-50)"), (p_dl, "#1baf7a", "p = 0.9 (định lý 1.1)"),
                     (p_gpt2, "#eb6834", "p = 0.80 (GPT-2)")):
    ax.semilogx(cac_G, amdahl(p, cac_G), color=mau, lw=2, label=nhan)
    ax.axhline(1 / (1 - p), color=mau, lw=1, ls=":")
    ax.text(1.3, 1 / (1 - p) + 0.4, f"trần {1 / (1 - p):.0f} lần", color=mau, fontsize=8)
for p, mau in ((p_resnet, "#2a78d6"), (p_gpt2, "#eb6834")):
    ax.plot(G, amdahl(p, G), "o", color=mau, ms=7, mec="white", mew=1.2)
    ax.annotate(f"G = 247: {amdahl(p, G):.1f} lần", (G, amdahl(p, G)), xytext=(8, -14),
                textcoords="offset points", fontsize=8)
ax.set_ylim(0, 22)
ax.set_xlabel("tăng tốc trên phần được tăng tốc, G (thang log)")
ax.set_ylabel("tăng tốc cả ứng dụng, S")
ax.set_title("Định luật Amdahl: G lớn đến đâu, S cũng dừng ở 1 / (1 − p)")
ax.legend(loc="center left", bbox_to_anchor=(0.0, 0.62), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Checkpoint 1.1 của sách đặt hai câu hỏi tính được. Câu thứ nhất: vì sao một GPU nhanh hơn 1,000
lần có thể chỉ làm huấn luyện nhanh hơn 5 lần khi khâu nạp dữ liệu chậm? Câu thứ hai: với
$p = 0.95$ và $G = 100$, cả ứng dụng nhanh lên bao nhiêu, so với trần khi $G$ vô hạn? Hãy tự tính
trước, rồi chạy ô dưới.

In [ ]:
G_1000, S_5 = sach(1000, nguon="1,000 · 5 lần · 0.95 · 100"), sach(5, nguon="1,000 · 5 lần · 0.95 · 100")
p_can = (1 - 1 / S_5) / (1 - 1 / G_1000)           # giải S(p, G) = S theo p
assert abs(amdahl(p_can, G_1000) - S_5) < 1e-12
print(f"câu 1: S = 5 với G = 1,000 nghĩa là p = {p_can:.4f}, tức {1 - p_can:.1%} thời gian nằm ngoài GPU")
p95, G100 = sach(0.95, nguon="1,000 · 5 lần · 0.95 · 100"), sach(100, nguon="1,000 · 5 lần · 0.95 · 100")
print(f"câu 2: S(0.95, 100) = {amdahl(p95, G100):.1f} lần, trần {1 / (1 - p95):.0f} lần;"
      f" G = 100 đã lấy được {amdahl(p95, G100) / (1 / (1 - p95)):.0%} của trần")

### Một ngân sách thời gian theo từng khâu

Phần này phỏng theo phần D của phòng thí nghiệm đi kèm chương. Một yêu cầu suy luận đi qua năm
khâu: việc của máy chủ chủ, truyền dữ liệu sang thiết bị, khởi chạy kernel, chính kernel, rồi hậu
xử lý. Một kernel nhanh hơn chỉ chia nhỏ khâu thứ tư. Các thời gian dưới đây là **giả định của sổ
tay**, không phải số của sách; bạn có thể đổi chúng.

In [ ]:
KHAU = {"máy chủ chủ": 2.0, "truyền dữ liệu": 1.5, "khởi chạy": 0.5, "kernel": 10.0, "hậu xử lý": 1.0}   # ms, giả định
tong = sum(KHAU.values())
p_kernel = KHAU["kernel"] / tong
for nhanh in (2, 10, 50):
    moi = dict(KHAU, kernel=KHAU["kernel"] / nhanh)
    S = tong / sum(moi.values())
    assert abs(S - amdahl(p_kernel, nhanh)) < 1e-12     # ngân sách theo khâu chính là Amdahl
    print(f"kernel nhanh hơn {nhanh:>2} lần: {tong:.1f} ms → {sum(moi.values()):.2f} ms, cả yêu cầu nhanh hơn {S:.2f} lần")
print(f"kernel chiếm p = {p_kernel:.1%} thời gian, nên trần là {1 / (1 - p_kernel):.2f} lần")

In [ ]:
mau_khau = ["#4a3aa7", "#eb6834", "#1baf7a", "#2a78d6", "0.6"]
fig, ax = plt.subplots(figsize=(7.5, 2.9))
cac_dong = [("ban đầu", KHAU), ("kernel nhanh 10 lần", dict(KHAU, kernel=KHAU["kernel"] / 10))]
for i, (ten, khau) in enumerate(cac_dong):
    trai = 0.0
    for (k, v), m in zip(khau.items(), mau_khau):
        ax.barh(i, v, left=trai, color=m, height=0.55, edgecolor="white", linewidth=1.5,
                label=k if i == 0 else None)
        trai += v
    ax.text(trai + 0.2, i, f"{trai:.1f} ms", va="center", fontsize=8)
ax.set_yticks([0, 1], [ten for ten, _ in cac_dong])
ax.invert_yaxis()
ax.set_xlim(0, tong * 1.12)
ax.set_xlabel("thời gian một yêu cầu (ms, giả định của sổ tay)")
ax.set_title(f"Kernel nhanh hơn 10 lần, cả yêu cầu chỉ nhanh hơn {tong / sum(cac_dong[1][1].values()):.1f} lần")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=5, fontsize=8, frameon=False)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 3. Năng lượng: di chuyển dữ liệu đắt hơn tính

Hình 9 của sách, theo các ước lượng 45 nm của Horowitz (2014), cho hai tỉ lệ: một lần truy cập DRAM
ngoài chip tốn khoảng 128 lần một lần đọc SRAM 8 KB, và hơn 20,000 lần một phép cộng INT8. Định
nghĩa 1.6 nói một lần truy cập DRAM tốn khoảng 200 lần một phép nhân cộng dồn số nguyên ngay tại
chỗ. Phần Ngộ nhận dùng một mô hình khác: 640 pJ cho một lần truy cập DRAM, 0.5 pJ cho một lần
truy cập SRAM L1 trên chip. Sách nói rõ các giá trị này không phải thông số chung của mọi thiết bị.

In [ ]:
dram_pJ = sach(640, nguon="640 pJ · 0.5 pJ")
l1_pJ = sach(0.5, nguon="640 pJ · 0.5 pJ")
print(f"mô hình của phần Ngộ nhận: DRAM ÷ L1 = {dram_pJ / l1_pJ:,.0f} lần")
ti_sram = sach(128, nguon="128× · hơn 20,000×")
ti_int8 = sach(20_000, nguon="128× · hơn 20,000×")
ti_mac = sach(200, nguon="45 nm · 200 lần")
print(f"hình 9: DRAM ÷ đọc SRAM 8 KB = {ti_sram}, DRAM ÷ cộng INT8 > {ti_int8:,}; định nghĩa 1.6: DRAM ÷ MAC số nguyên ≈ {ti_mac}")
print(f"nếu đặt DRAM ở {dram_pJ} pJ, tỉ lệ 128 của hình 9 cho một lần đọc SRAM 8 KB khoảng {dram_pJ / ti_sram:.0f} pJ,"
      f" gấp {dram_pJ / ti_sram / l1_pJ:.0f} lần con số {l1_pJ} pJ cho L1")

Dòng cuối là một phép trộn mà bạn không nên làm khi tính thật. Hai con số nói về hai bộ nhớ khác
nhau (SRAM 8 KB và L1), đến từ hai chỗ khác nhau của chương, và sách không nói chúng thuộc cùng một
mô hình. Điều chung của cả ba nguồn là khoảng cách: một lần đi ra DRAM đắt hơn từ hai tới bốn bậc
độ lớn so với một lần dùng dữ liệu đã nằm gần, hay so với chính phép tính.

Áp vào mô hình của napkin math 1.1 (640 pJ mỗi lần chạm DRAM, 1 pJ mỗi phép tính): nếu mỗi lần
chạm DRAM phục vụ $r$ phép, năng lượng mỗi phép là $640/r + 1$ pJ. Phần dành cho di chuyển dữ liệu
chỉ xuống dưới một nửa khi $r$ vượt 640.

**Dự đoán:** mảng systolic 128 × 128 của napkin math 1.1 tốn khoảng 11 pJ mỗi phép. Bao nhiêu
phần trong số đó vẫn là di chuyển dữ liệu?

In [ ]:
tinh_pJ = sach(1, nguon="4 · 640 · 1 · 2561")
lan_systolic = sach(2, nguon="2 · 0.016 · 11") / sach(128, nguon="128 · 128×128")
E_sys = lan_systolic * dram_pJ + tinh_pJ
theo_sach("mảng systolic (pJ mỗi phép)", E_sys, sach=11, nguon="2 · 0.016 · 11")
print(f"trong {E_sys:.0f} pJ có {lan_systolic * dram_pJ:.0f} pJ di chuyển dữ liệu, tức {lan_systolic * dram_pJ / E_sys:.0%}")
print(f"mốc ngây thơ: {4 * dram_pJ:,.0f} trong {4 * dram_pJ + tinh_pJ:,.0f} pJ, tức {4 * dram_pJ / (4 * dram_pJ + tinh_pJ):.2%}")

In [ ]:
r = np.logspace(-1, 4, 300)
phan = (dram_pJ / r) / (dram_pJ / r + tinh_pJ)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogx(r, 100 * phan, color="#2a78d6", lw=2)
for rr, nhan, (dx, dy), ha in ((0.25, "mốc ngây thơ: 4 lần chạm mỗi phép", (8, -16), "left"),
                               (1 / lan_systolic, "mảng systolic 128 × 128", (8, 4), "left"),
                               (dram_pJ / tinh_pJ, "di chuyển = tính", (-8, -16), "right")):
    y = 100 * (dram_pJ / rr) / (dram_pJ / rr + tinh_pJ)
    ax.plot(rr, y, "o", color="#eb6834", ms=6, mec="white", mew=1.2)
    ax.annotate(f"{nhan}\nr = {rr:g}: {y:.0f} %", (rr, y), xytext=(dx, dy), textcoords="offset points",
                fontsize=8, ha=ha, va="top" if dy < 0 else "bottom")
ax.set_ylim(0, 110)
ax.set_xlabel("số phép tính mỗi lần chạm DRAM, r (thang log)")
ax.set_ylabel("phần năng lượng cho di chuyển (%)")
ax.set_title("Mô hình 640 pJ / 1 pJ của napkin math 1.1")
plt.tight_layout()
plt.show()

Ngay cả mảng systolic, với lợi thế 232.8 lần so với mốc ngây thơ, vẫn tiêu khoảng 10 trong 11 pJ
mỗi phép cho việc đưa dữ liệu vào. Đây là lý do chương coi tái dùng trọng số và chia khối là việc
của năng lượng chứ không chỉ của tốc độ.

### Hành trình của một tensor, và giới hạn của tốc độ ánh sáng

Mô hình đại diện 1.2 của sách theo một giây âm thanh của bài toán phát hiện từ khoá qua phân cấp bộ
nhớ của một GPU. Các con số đều là xấp xỉ. Napkin math 1.2 thì hỏi vì sao không đọc thẳng mọi thứ
từ HBM: trên một đế cỡ H100, tín hiệu đi khoảng 20 mm.

In [ ]:
mau_am, byte_moi = sach(16_000, nguon="16,000 mẫu × 2 byte = 32 KB"), sach(2, nguon="16,000 mẫu × 2 byte = 32 KB")
kich_thuoc = mau_am * byte_moi
theo_sach("tensor một giây âm thanh, FP16 (KB)", kich_thuoc / 1e3, sach=32, nguon="16,000 mẫu × 2 byte = 32 KB")

GHz = sach(2, nguon="2 GHz · 500 ps")
theo_sach("một chu kỳ ở 2 GHz (ps)", 1e12 / (GHz * 1e9), sach=500, nguon="2 GHz · 500 ps")
for ten, ns, nguon in (("HBM", sach(300, nguon="~300 ns · ~20 pJ/bit", tol=50), "~300 ns · ~20 pJ/bit"),
                       ("L2", sach(4, nguon="~4 ns · ~1 ns", tol=1), "~4 ns · ~1 ns"),
                       ("L1 hoặc bộ nhớ chia sẻ", sach(1, nguon="~4 ns · ~1 ns", tol=0.5), "~4 ns · ~1 ns")):
    print(f"{ten:>22}: khoảng {ns:g} ns = {ns * GHz:,.0f} chu kỳ ở {GHz} GHz")

pJ_bit_hbm = sach(20, nguon="~300 ns · ~20 pJ/bit", tol=5)
pJ_bit_reg = sach(0.1, nguon="312 TFLOP/s · ~0.1 pJ/bit", tol=0.05)
E_hbm = kich_thuoc * 8 * pJ_bit_hbm
E_reg = kich_thuoc * 8 * pJ_bit_reg
print(f"đưa 32 KB từ HBM: {E_hbm / 1e6:.2f} µJ; cùng lượng bit ở thanh ghi: {E_reg / 1e3:.1f} nJ;"
      f" chênh {E_hbm / E_reg:.0f} lần")

C_ANH_SANG = 299_792_458                   # m/s, hằng số vật lý
duong_mm = sach(20, nguon="814 mm² · ~20 mm", tol=5)
t_lan_truyen = duong_mm * 1e-3 / (0.5 * C_ANH_SANG)
theo_sach("20 mm ở nửa tốc độ ánh sáng (ps)", t_lan_truyen * 1e12, sach=130, nguon="≈ 0.5c · ≈ 130 ps", tol=5)
chu_ky_hbm = sach(100, nguon="100+ chu kỳ")
print(f"lan truyền chỉ {t_lan_truyen * 1e12:.0f} ps, chưa tới một chu kỳ; một lần truy cập HBM là {chu_ky_hbm}+ chu kỳ,"
      f" tức hơn {chu_ky_hbm / (GHz * 1e9) * 1e9:.0f} ns")

Sách tự rút ra điều hơi bất ngờ: 20 mm chỉ tốn khoảng 130 ps, chưa tới một chu kỳ 500 ps, nên
riêng khoảng cách không chứng minh được rằng một lần truy cập trong một chu kỳ là không thể. Phần
đắt là cả con đường: độ trễ của mảng DRAM, tuần tự hoá, giao thức, liên kết và hàng đợi. Đó là lý
do thanh ghi và SRAM phải nằm sát đơn vị tính.

## 4. Đệm kép: giấu thời gian chép sau thời gian tính

Môi trường chạy giấu thời gian truyền bằng đệm kép (double buffering): trong lúc kernel tính batch
$n$, bộ chép đưa batch $n + 1$ vào bộ đệm thứ hai. Ta mô hình hoá nó như hai tài nguyên, một bộ
chép và một bộ tính, mỗi tài nguyên làm một việc một lúc. Một batch chỉ được chép vào một bộ đệm
khi batch trước đó dùng bộ đệm ấy đã tính xong.

Với một bộ đệm, chép và tính phải xen kẽ nhau: $N(t_\text{chép} + t_\text{tính})$. Với hai bộ đệm,
tài nguyên chậm hơn đặt nhịp: $t_\text{chép} + (N - 1)\max(t_\text{chép}, t_\text{tính}) +
t_\text{tính}$. Hàm dưới không dùng công thức nào trong hai công thức ấy, chỉ lập lịch từng việc,
rồi kiểm rằng kết quả trùng với cả hai.

In [ ]:
def lap_lich(n_batch, t_chep, t_tinh, so_dem):
    """Lịch của bộ chép và bộ tính; trả về (bắt đầu chép, xong chép, bắt đầu tính, xong tính)."""
    bd_chep, xong_chep, bd_tinh, xong_tinh = [], [], [], []
    for i in range(n_batch):
        bd = max(xong_chep[i - 1] if i else 0.0,                      # bộ chép rảnh
                 xong_tinh[i - so_dem] if i >= so_dem else 0.0)       # bộ đệm của batch này đã trống
        bd_chep.append(bd)
        xong_chep.append(bd + t_chep)
        bd2 = max(xong_chep[i], xong_tinh[i - 1] if i else 0.0)       # dữ liệu đã tới, bộ tính rảnh
        bd_tinh.append(bd2)
        xong_tinh.append(bd2 + t_tinh)
    return bd_chep, xong_chep, bd_tinh, xong_tinh


for t_chep, t_tinh in ((3.0, 5.0), (5.0, 3.0), (4.0, 4.0)):
    for N in (1, 2, 5, 20):
        assert lap_lich(N, t_chep, t_tinh, 1)[3][-1] == N * (t_chep + t_tinh)
        assert lap_lich(N, t_chep, t_tinh, 2)[3][-1] == t_chep + (N - 1) * max(t_chep, t_tinh) + t_tinh
print("lịch từng việc trùng với cả hai công thức, ở cả ba trường hợp chép < tính, chép > tính, chép = tính")

T_CHEP, T_TINH, N_BATCH = 3.0, 5.0, 5          # ms, giả định của sổ tay
for so_dem in (1, 2):
    xong = lap_lich(N_BATCH, T_CHEP, T_TINH, so_dem)[3][-1]
    print(f"{so_dem} bộ đệm: {N_BATCH} batch xong sau {xong:.0f} ms")

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 3.6), sharex=True)
for ax, so_dem in zip(axs, (1, 2)):
    bc, xc, bt, xt = lap_lich(N_BATCH, T_CHEP, T_TINH, so_dem)
    for i in range(N_BATCH):
        ax.barh(1, xc[i] - bc[i], left=bc[i], color="#eb6834", height=0.6, edgecolor="white", linewidth=1.5)
        ax.barh(0, xt[i] - bt[i], left=bt[i], color="#2a78d6", height=0.6, edgecolor="white", linewidth=1.5)
        ax.text((bc[i] + xc[i]) / 2, 1, str(i), ha="center", va="center", fontsize=8, color="white")
        ax.text((bt[i] + xt[i]) / 2, 0, str(i), ha="center", va="center", fontsize=8, color="white")
    ax.set_yticks([0, 1], ["tính", "chép"])
    ax.set_title(f"{so_dem} bộ đệm: xong sau {xt[-1]:.0f} ms", fontsize=9, loc="left")
    ax.grid(axis="y", visible=False)
axs[1].set_xlabel("thời gian (ms, giả định của sổ tay: chép 3 ms, tính 5 ms mỗi batch); số trong ô là batch")
plt.tight_layout()
plt.show()

Với hai bộ đệm, bộ tính gần như không bao giờ chờ: thời gian chép đã nằm gọn dưới thời gian tính.
Nếu chép lâu hơn tính, đệm kép vẫn giúp, nhưng khi ấy bộ tính phải chờ và việc chép đặt nhịp. Đây
cũng là ý của định luật Amdahl theo một cách khác: chồng hai việc lên nhau chỉ giấu được việc ngắn
hơn, không giấu được việc dài hơn.

## 5. Nhiều chip: AllReduce và vì sao tám GPU không nhanh gấp tám

Khi huấn luyện song song dữ liệu, mỗi bước mọi GPU phải cộng gradient của nhau lại. Cách làm phổ
biến là AllReduce vòng: các GPU xếp thành một vòng, mảng gradient chia thành $N$ phần, và qua
$2(N - 1)$ bước mỗi GPU gửi một phần cho GPU kế tiếp. Nửa đầu cộng dồn, nửa sau phát kết quả cho
tất cả. Ta chạy thật thuật toán này trên số nguyên và đếm số phần tử mỗi GPU gửi đi.

In [ ]:
def allreduce_vong(cac_mang):
    """AllReduce vòng trên N mảng; trả về kết quả ở từng GPU và số phần tử mỗi GPU đã gửi."""
    N = len(cac_mang)
    dem = [m.copy().reshape(N, -1) for m in cac_mang]
    da_gui = [0] * N
    for buoc in range(N - 1):                       # cộng dồn: sau N − 1 bước, mỗi phần nằm đủ ở một GPU
        thu = [(r, (r - buoc) % N, dem[r][(r - buoc) % N].copy()) for r in range(N)]
        for r, phan, v in thu:
            dem[(r + 1) % N][phan] += v
            da_gui[r] += v.size
    for buoc in range(N - 1):                       # phát lại: phần đã cộng đủ đi hết một vòng
        thu = [(r, (r + 1 - buoc) % N, dem[r][(r + 1 - buoc) % N].copy()) for r in range(N)]
        for r, phan, v in thu:
            dem[(r + 1) % N][phan] = v
            da_gui[r] += v.size
    return [d.reshape(-1) for d in dem], da_gui


rng = np.random.default_rng(0)
for N in (2, 3, 8, 16):
    tai = 4 * N                                     # phần tử gradient mỗi GPU
    cac_mang = [rng.integers(-9, 10, tai) for _ in range(N)]
    ket_qua, da_gui = allreduce_vong(cac_mang)
    assert all(np.array_equal(k, sum(cac_mang)) for k in ket_qua)
    assert all(g * N == 2 * (N - 1) * tai for g in da_gui)
    print(f"N = {N:>2}: mọi GPU có đúng tổng; mỗi GPU gửi {da_gui[0]} phần tử = {da_gui[0] / tai:.4g} lần tải gradient")

Mỗi GPU gửi $2(N - 1)/N$ lần tải gradient, tiến dần tới 2 khi $N$ lớn. Giờ ta tính kịch bản trong
phần Ngộ nhận của sách: tám GPU, 1 GB gradient, NVLink 600 GB/s hai chiều, tức một nửa mỗi chiều,
và một bước tính 50 ms.

**Dự đoán:** nếu không chồng truyền thông lên tính toán, tám GPU nhanh hơn một GPU bao nhiêu lần?

In [ ]:
N8 = sach(8, nguon="tám GPU · 8×")
he_so = 2 * (N8 - 1) / N8
theo_sach("tải mỗi GPU, tính theo lần gradient", he_so, sach=1.75,
          nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm")
bw_moi_chieu = sach(600, nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm") / 2                      # GB/s
gradient_GB = sach(1, nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm")
t_tinh_ms = sach(50, nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm")


def t_allreduce_ms(N, bw_GBs, tai_GB=gradient_GB):
    return 2 * (N - 1) / N * tai_GB / bw_GBs * 1e3


t8 = t_allreduce_ms(N8, bw_moi_chieu)
theo_sach("thời gian AllReduce (ms)", t8, sach=5.83,
          nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm")
theo_sach("so với bước tính (%)", 100 * t8 / t_tinh_ms, sach=11.7,
          nguon="1.75× · 600 GB/s · 1 GB · 5.83 ms · 50 ms · 11.7 phần trăm")
tang_toc = N8 * t_tinh_ms / (t_tinh_ms + t8)
theo_sach("tăng tốc tám GPU, không chồng", tang_toc, sach=7.2, nguon="7.2× · 89.6 phần trăm")
theo_sach("hiệu suất mở rộng (%)", 100 * tang_toc / N8, sach=89.6, nguon="7.2× · 89.6 phần trăm")

Con số 89.6 % là hiệu suất mở rộng (scaling efficiency): tăng tốc chia cho số GPU. Sách thêm rằng
đây là trước khi tính phụ trội của giao thức, mất cân bằng tải và các rào đồng bộ,
nên con số thật còn thấp hơn. Hình dưới mở rộng kịch bản theo $N$ trong hai trường hợp. Trong một
máy, vòng đi qua NVLink. Khi vòng phải ra ngoài máy, liên kết chậm nhất đặt nhịp; ta lấy 50 GB/s,
mức của một card mạng trong chuỗi băng thông của chương, và **giả định** đó là băng thông mỗi
chiều. Đường "chồng hoàn hảo" giả định truyền thông nằm gọn dưới 50 ms tính toán, nên nó đúng bằng N.

In [ ]:
bw_mang = sach(50, nguon="3,350 · 900 · 64 · 50 GB/s")
N_may = np.arange(1, 9)
N_cum = 2 ** np.arange(0, 7)


def tang_toc_khong_chong(N, bw):
    return np.array([n * t_tinh_ms / (t_tinh_ms + t_allreduce_ms(n, bw)) for n in N])


def tang_toc_chong(N, bw):
    return np.array([n * t_tinh_ms / max(t_tinh_ms, t_allreduce_ms(n, bw)) for n in N])


fig, ax = plt.subplots(figsize=(7.5, 4.2))
chong = tang_toc_chong(N_cum, bw_mang)
dung_N = np.array_equal(chong, N_cum)                       # đúng khi truyền luôn ngắn hơn bước tính
ax.plot(N_cum, chong, "s-", color="#1baf7a", ms=5,
        label="card mạng 50 GB/s, chồng hoàn hảo" + (": đúng N lần" if dung_N else ""))
ax.plot(N_cum, tang_toc_khong_chong(N_cum, bw_mang), "^-", color="#eb6834", ms=6, label="card mạng 50 GB/s, không chồng")
ax.plot(N_may, tang_toc_khong_chong(N_may, bw_moi_chieu), "o-", color="#2a78d6", ms=5, label="NVLink trong một máy, không chồng")
ax.annotate(f"NVLink, 8 GPU: {tang_toc:.1f} lần", (8, tang_toc), xytext=(-30, 22), textcoords="offset points",
            fontsize=8, ha="right", arrowprops=dict(arrowstyle="-", color="0.55", lw=0.8, shrinkB=4))
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xticks(N_cum, [str(n) for n in N_cum])
ax.set_yticks(N_cum, [str(n) for n in N_cum])
ax.set_xlabel("số GPU, N")
ax.set_ylabel("tăng tốc so với một GPU")
ax.set_title("AllReduce vòng, 1 GB gradient, bước tính 50 ms")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

for N in (8, 64):
    print(f"N = {N:>2}, card mạng: AllReduce {t_allreduce_ms(N, bw_mang):.1f} ms,"
          f" không chồng {tang_toc_khong_chong([N], bw_mang)[0]:.1f} lần, chồng {tang_toc_chong([N], bw_mang)[0]:.1f} lần")

Không chồng thì mỗi bước đều phải trả trọn thời gian AllReduce, và qua card mạng khoản ấy đã lên tới
khoảng 70 % bước tính. Chồng truyền thông lên tính toán là thứ lấy lại gần như toàn bộ, miễn là thời gian
truyền còn ngắn hơn thời gian tính, đúng như bài học của đệm kép ở phần 4.

### Vì sao phải nhiều chip: quy mô của GPT-3

Sách lấy ước tính $3.14 \cdot 10^{23}$ phép dấu phẩy động để huấn luyện GPT-3, chia cho tốc độ FP8
đỉnh 1.98 PFLOP/s của một H100. Sách nói rõ đây là một phép so sánh quy mô, không phải tái hiện quá
trình huấn luyện GPT-3: độ chính xác, phần cứng và tải công việc đều là giả định.

In [ ]:
FLOP_gpt3 = sach(3.14e23, nguon="3.14 · 10^23")
R_fp8 = sach(1.98, nguon="1.98 PFLOP/s") * 1e15
NAM = 365 * 24 * 3600
theo_sach("cận dưới trên một thiết bị (năm)", FLOP_gpt3 / R_fp8 / NAM, sach=5,
          nguon="5 năm · 8.4–12.6 năm · 40–60 phần trăm")
u_thap = sach(40, nguon="5 năm · 8.4–12.6 năm · 40–60 phần trăm") / 100
u_cao = sach(60, nguon="5 năm · 8.4–12.6 năm · 40–60 phần trăm") / 100
theo_sach("ở mức sử dụng 60 % (năm)", FLOP_gpt3 / (R_fp8 * u_cao) / NAM, sach=8.4,
          nguon="5 năm · 8.4–12.6 năm · 40–60 phần trăm")
theo_sach("ở mức sử dụng 40 % (năm)", FLOP_gpt3 / (R_fp8 * u_thap) / NAM, sach=12.6,
          nguon="5 năm · 8.4–12.6 năm · 40–60 phần trăm")
for u in (u_cao, u_thap):
    can = FLOP_gpt3 / (R_fp8 * u * 30 * 24 * 3600)
    print(f"để xong trong 30 ngày ở mức sử dụng {u:.0%}: cần {math.ceil(can)} thiết bị, nếu mở rộng tuyến tính hoàn hảo")

Dòng cuối là phép tính của sổ tay, không phải của sách, và nó giả định điều mà hình ở trên vừa cho
thấy là không có: hiệu suất mở rộng 100 %. Với mỗi điểm phần trăm mất đi cho truyền thông, số thiết
bị cần thêm tăng theo.

## 6. Napkin math 1.10: năng lượng và cacbon của một đội máy

Hộp napkin math 1.10 là một phép tính độ nhạy minh hoạ, không phải kết quả mua sắm đã đo. Nó coi
thông lượng đỉnh là thông lượng hữu ích, cho cả hai đội chạy liên tục ở công suất ghi trên giấy, và
bỏ qua máy chủ, làm mát và phát thải nội hàm.

In [ ]:
P_cpu = sach(100, nguon="100 W · 1 TFLOP/s · 0.01 TFLOP/s mỗi oát")
R_cpu = sach(1, nguon="100 W · 1 TFLOP/s · 0.01 TFLOP/s mỗi oát")
P_npu = sach(5, nguon="5 W · 10 TFLOP/s · 2 TFLOP/s mỗi oát")
R_npu = sach(10, nguon="5 W · 10 TFLOP/s · 2 TFLOP/s mỗi oát")
theo_sach("CPU: TFLOP/s mỗi oát", R_cpu / P_cpu, sach=0.01, nguon="100 W · 1 TFLOP/s · 0.01 TFLOP/s mỗi oát")
theo_sach("NPU: TFLOP/s mỗi oát", R_npu / P_npu, sach=2, nguon="5 W · 10 TFLOP/s · 2 TFLOP/s mỗi oát")
theo_sach("khoảng cách hiệu suất đỉnh", (R_npu / P_npu) / (R_cpu / P_cpu), sach=200, nguon="200×")

so_cpu = sach(1000, nguon="1,000 máy chủ · 100 W · 24 giờ · 2,400 kWh mỗi ngày")
so_npu = sach(100, nguon="100 chip · 5 W · 24 giờ · 12 kWh mỗi ngày")
gio = sach(24, nguon="1,000 máy chủ · 100 W · 24 giờ · 2,400 kWh mỗi ngày")
E_cpu = so_cpu * P_cpu * gio / 1e3                    # kWh mỗi ngày
E_npu = so_npu * P_npu * gio / 1e3
theo_sach("đội CPU (kWh mỗi ngày)", E_cpu, sach=2400, nguon="1,000 máy chủ · 100 W · 24 giờ · 2,400 kWh mỗi ngày")
theo_sach("đội NPU (kWh mỗi ngày)", E_npu, sach=12, nguon="100 chip · 5 W · 24 giờ · 12 kWh mỗi ngày")
assert so_cpu * R_cpu == so_npu * R_npu               # hai đội cho cùng thông lượng: 1,000 TFLOP/s
kg_kWh = sach(0.429, nguon="0.429 kg mỗi kWh · 373.9 tấn mỗi năm")
tiet_kiem = (E_cpu - E_npu) * 365 * kg_kWh / 1e3       # tấn mỗi năm
theo_sach("khí cacbonic tiết kiệm (tấn mỗi năm)", tiet_kiem, sach=373.9,
          nguon="0.429 kg mỗi kWh · 373.9 tấn mỗi năm")

suy_luan = sach(1e9, nguon="1 tỷ lần suy luận mỗi ngày")
print(f"suy ra (sách không nói): mỗi lần suy luận được {so_cpu * R_cpu * 1e12 * 86_400 / suy_luan / 1e9:.1f} GFLOP")

Hai đội cho đúng cùng thông lượng, 1,000 TFLOP/s, nên tỉ số năng lượng của chúng đúng bằng tỉ số
hiệu năng trên mỗi oát, 200 lần. Nhưng sách cảnh báo rằng chuyên biệt hoá chỉ tiết kiệm khi tải
công việc đi được đường hiệu quả của chip. Ta thử điều đó: giả sử tải công việc chỉ lấy được một
phần $u$ của thông lượng đỉnh trên NPU so với trên CPU, tức đội NPU phải lớn gấp $1/u$.

**Dự đoán:** $u$ phải nhỏ tới đâu thì khoản tiết kiệm biến mất?

In [ ]:
cac_u = np.logspace(-3, 0, 300)
tiet_kiem_u = (E_cpu - E_npu / cac_u) * 365 * kg_kWh / 1e3
u_hoa_von = E_npu / E_cpu
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogx(cac_u, tiet_kiem_u, color="#2a78d6", lw=2)
ax.axhline(0, color="0.4", lw=1)
ax.plot(1, tiet_kiem, "o", color="#2a78d6", ms=6, mec="white", mew=1.2)
ax.annotate(f"kịch bản của sách: {tiet_kiem:.1f} tấn", (1, tiet_kiem), xytext=(-10, -18),
            textcoords="offset points", fontsize=8, ha="right")
ax.plot(u_hoa_von, 0, "o", color="#eb6834", ms=6, mec="white", mew=1.2)
ax.annotate(f"hoà vốn: u = {u_hoa_von:.3f}", (u_hoa_von, 0), xytext=(10, 14), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.55", lw=0.8))
ax.set_ylim(-150, 450)
ax.set_xlabel("phần thông lượng đỉnh mà NPU thật sự giao, so với CPU, u (thang log)")
ax.set_ylabel("tấn khí cacbonic tiết kiệm mỗi năm")
ax.set_title("Napkin math 1.10: độ nhạy theo mức sử dụng của NPU")
plt.tight_layout()
plt.show()

Đường cong rất phẳng gần $u = 1$, vì khoản tiết kiệm đến gần hết từ việc tắt 2,400 kWh của đội CPU.
Ngay cả khi NPU chỉ giao được 10 % thông lượng đỉnh, khoản tiết kiệm vẫn còn hơn 95 %. Nó chỉ sụp
khi $u$ xuống cỡ 0.005, tức khi tải công việc gần như không đi được đường của NPU, chẳng hạn vì phép
toán không được hỗ trợ phải chạy lại trên CPU. Sách kết hộp bằng lời dặn: trước khi dùng kết quả
cho mua sắm hay tính phát thải, phải thay giả định bằng số yêu cầu mỗi giây và công suất đo tại ổ
điện.

In [ ]:
assert (E_cpu - E_npu / 0.1) / (E_cpu - E_npu) > 0.95        # ở u = 10 %, còn hơn 95 % khoản tiết kiệm

## Thử thêm

1. Ở ô `stages`, giữ nguyên tổng thời gian nhưng chuyển 1 ms từ "máy chủ chủ" sang "kernel". Trần
   của tăng tốc thay đổi thế nào? Khâu nào bạn sẽ tối ưu trước nếu kernel đã nhanh hơn 10 lần?
2. Thêm dòng `t_tinh_ms = 10` ở đầu ô `plot-scaling`, như khi mô hình nhỏ hơn hay batch mỗi GPU nhỏ
   hơn. Hiệu suất mở rộng của tám GPU qua NVLink còn bao nhiêu phần trăm, và đường "chồng hoàn hảo"
   còn bằng N không?
3. Ở ô `napkin110`, thêm dòng `kg_kWh = 0.05` ngay dưới dòng gán `kg_kWh`, như một lưới điện nhiều
   thuỷ điện. Khoản tiết kiệm tính bằng tấn thay đổi theo tỉ lệ nào, còn tỉ số năng lượng giữa hai
   đội có đổi không?

## Tóm lại

* Định luật Amdahl: phần $1 - p$ không được tăng tốc đặt trần $1/(1 - p)$; 247 lần trên đường nhân
  ma trận thành 18.6 lần cho ResNet-50 và 4.9 lần cho GPT-2 trong kịch bản của sách.
* Trong mô hình 640 pJ cho một lần chạm DRAM và 1 pJ cho một phép tính, di chuyển dữ liệu chiếm phần
  lớn năng lượng cho tới khi mỗi lần chạm DRAM phục vụ hàng trăm phép; cả mảng systolic cũng vậy.
* Đệm kép và chồng truyền thông lên tính toán chỉ giấu được việc ngắn hơn trong hai việc.
* AllReduce vòng bắt mỗi GPU gửi $2(N - 1)/N$ lần tải gradient; không chồng, tám GPU của sách được 7.2
  lần, và qua một liên kết chậm hơn thì còn ít hơn nhiều.
* Napkin math 1.10 tiết kiệm khoảng 373.9 tấn mỗi năm trong kịch bản minh hoạ, và khoản đó vững
  trước mức sử dụng của NPU, trừ khi tải công việc gần như không đi được đường của chip.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tăng tốc phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch11-hw-acceleration/), dựng từ chương
[*Hardware Acceleration*](https://mlsysbook.ai/vol1/hw_acceleration/hw_acceleration.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_11_hw_accel.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.